# Trotter Optimization Comparison: `data` vs `data_new_sign`

This notebook allows you to interactively compare Trotter ansatz optimization performance, ground state overlaps, **energy expectations for any state in `:spin_first` sign convention**, and depth scaling (`num_exponentials`) between the **`data`** (JLD2) and **`data_new_sign`** (HDF5) datasets.

In [1]:
using Lattices
using LinearAlgebra
using SparseArrays
using JLD2
using HDF5
using Test

# Include project code
if !isdefined(Main, :UtilityFunctions)
    include("utility_functions.jl")
end
using .UtilityFunctions
include("trotter.jl")
using .Trotter
include("ed_objects.jl")
include("ed_functions.jl")

build_save_name_prefix

## 1. Inspect Data, Reference States & Energy Expectations

Compare $U$-values, selected sector, Slater reference state overlaps, and energy expectation values across $U$ values for both datasets.

In [3]:
function compute_energy_expectation(state, u_val, H_hop, H_int)
    return real(state' * (H_hop + u_val * H_int) * state / (state'*state))
end

folder_data = "/home/jek354/research/data/new_data/data/N=(3, 2)_3x3"
folder_new_sign = "/home/jek354/research/data/new_data/data_new_sign/N=(3, 2)_3x3"

# Load data dataset
U1, vecs_data, indexer_data, _, N_elec1, _, _, _ = load_ED_data(
    folder_data; verbose=false, sign_convention=:spin_first, use_slater_reference="slater"
)
slater_data = vecs_data[1, :]
Lvec1 = parse_lattice_dimension(folder_data)
# H_hop1, _, _ = Trotter.TamFermion.HubbardMomentumBasis(1.0, 0.0, Lvec1, N_elec1; indexer=indexer_data)
# H_int1, _, _ = Trotter.TamFermion.HubbardMomentumBasis(0.0, 1.0, Lvec1, N_elec1; indexer=indexer_data)

# Load data_new_sign dataset
U2, vecs_new, indexer_new, _, N_elec2, _, _, _ = load_ED_data(
    folder_new_sign; verbose=false, sign_convention=:spin_first, use_slater_reference="slater"
)
slater_new = vecs_new[1, :]
Lvec2 = parse_lattice_dimension(folder_new_sign)
# H_hop2, _, _ = Trotter.TamFermion.HubbardMomentumBasis(1.0, 0.0, Lvec2, N_elec2; indexer=indexer_new)
# H_int2, _, _ = Trotter.TamFermion.HubbardMomentumBasis(0.0, 1.0, Lvec2, N_elec2; indexer=indexer_new)

i = 21 # U index for data
u_val1 = U1[i]
gs_data = vecs_data[15, :]
# println("=== DATA (JLD2) at U = ", round(u_val1, digits=4), " ===")
# println("U range: ", U1[i], " to ", U1[end], " (total ", length(U1), " points)")
# println("Slater Energy Expectation: ", round(compute_energy_expectation(slater_data, u_val1, H_hop1, H_int1), digits=6))
# println("Ground State ED Energy:   ", round(compute_energy_expectation(gs_data, u_val1, H_hop1, H_int1), digits=6))
# println("Overlap <Slater|u(U)>:      ", round(abs(dot(slater_data, gs_data)), digits=6))
# println("True ground state at U:  ", eigvals(Matrix(H_hop1 + u_val1 * H_int1))[1])

j = 2 # U index for data_new_sign
u_val2 = U2[j]
gs_new = vecs_new[j, :]
# println("\n=== DATA_NEW_SIGN (HDF5) at U = ", round(u_val2, digits=4), " ===")
# println("U range: ", U2[1], " to ", U2[end], " (total ", length(U2), " points)")
# println("Slater Energy Expectation: ", round(compute_energy_expectation(slater_new, u_val2, H_hop2, H_int2), digits=6))
# println("Ground State ED Energy:   ", round(compute_energy_expectation(gs_new, u_val2, H_hop2, H_int2), digits=6))
# println("Overlap <Slater|u(U)>:      ", round(abs(dot(slater_new, gs_new)), digits=6))
# println("True ground state at U:  ", eigvals(Matrix(H_hop2 + u_val2 * H_int2))[1])


lattice = Square((3,2), Periodic())
subspace = HubbardSubspace(3, 2, lattice; k=indexer_new.k)
new_hopping = create_Hubbard(HubbardModel(1.0,0.0,0.0,false), subspace; indexer=indexer_new, momentum_basis=true, sign_convention=:spin_first)
new_interaction = create_Hubbard(HubbardModel(0.0,1.0,0.0,false), subspace; indexer=indexer_new, momentum_basis=true, sign_convention=:spin_first)
H = Matrix(new_hopping .+ u_val2 .* new_interaction)
println(eigvals(H)[1])
println(gs_new'*H*gs_new)

LoadError: IOError: readdir("/home/jek354/research/data/new_data/data/N=(3, 2)_3x3"): no such file or directory (ENOENT)

In [28]:
H = Trotter.Hubbard(1, 0.25, (3,2), (2,2), :real; use_pbc=true, returnBasis=false, q_target=nothing)
E, V = eigen(Matrix(H))
println(E)

LoadError: ArgumentError: q_target and basis_sector are not supported for real-space basis

In [ ]:
using Lattices
using LinearAlgebra
using SparseArrays
using JLD2
using HDF5
using Test
using Plots
# Include project code
if !isdefined(Main, :UtilityFunctions)
    include("utility_functions.jl")
end
using .UtilityFunctions
include("trotter.jl")
using .Trotter
include("ed_objects.jl")
include("ed_functions.jl")

folder_new_sign = "/home/jek354/research/data/new_data/data_h5_fixed/N=(3, 2)_3x3"

# # Load data_new_sign dataset
U2, vecs_new, indexer_new, _, N_elec2, _, _, sign_convention = load_ED_data(folder_new_sign; 
    sign_convention=:coordinate_first, verbose=false, use_slater_reference=false)
Lvec2 = parse_lattice_dimension(folder_new_sign)

# j = 2 # U index for data_new_sign
# u_val2 = U2[j]
# println("U=$u_val2")
# gs_new = vecs_new[j, :]
# println(sign_convention)
# lattice = Square(Tuple(Lvec2), Periodic())
# order = sign_convention == :spin_first ? ColSnake() : RowSnake()

# H, basis, counts = Trotter.HubbardMomentumBasis(1.0, u_val2, (3, 3), (3 ,2); indexer=indexer_new, sign_convention=sign_convention)
# energy,V = eigen(Matrix(H))
# println(energy)

# k = indexer_new.k
# subspace = HubbardSubspace(3,2, lattice; k=k)
# indexer_new = CombinationIndexer(subspace; order=order)
# println(length(indexer_new.inv_comb_dict), " ", length(gs_new))
# # println(indexer_new.inv_comb_dict[1:10])
# new_hopping = create_Hubbard(HubbardModel(1.0,0.0,0.0,false), subspace; 
#                     indexer=indexer_new, momentum_basis=true, sign_convention=sign_convention, lattice_ordering=order)
# new_interaction = create_Hubbard(HubbardModel(0.0,1.0,0.0,false), subspace; 
#                     indexer=indexer_new, momentum_basis=true, sign_convention=sign_convention, lattice_ordering=order)
# H = Matrix(new_hopping .+ u_val2 .* new_interaction)
# v = eigvecs(H)
# E = eigvals(H)[1:4]
# println("E=$E")
# println("overlap: $(v[:,1]'*gs_new)")
# println("overlap: $(V[:,1]'*gs_new)")
# println("pred energy: $(gs_new'*H*gs_new)")

# s1 = sortperm(abs.(v[:,1]))
# s2 = sortperm(abs.(gs_new))
# s3 = sortperm(abs.(V[:,1]))
# p = plot(abs2.(v[:,1])[s1], la=0.5,label="create_hubbard", yscale=:log10, legend=:bottomright, title=k)
# plot!(p, abs2.(gs_new)[s2],la=0.5 ,linestyle=:solid,label=".h5")
# plot!(p, abs2.(V[:,1])[s3],linestyle=:dash, la=0.5, label="TamFermion")
# display(p)




In [11]:
vecs_new

336×2 Matrix{ComplexF64}:
  0.000921371+0.00263232im   -0.00189544-0.00528932im
   0.00123301+6.97532e-5im   -0.00238419-7.15551e-5im
   0.00111661+0.0016871im    -0.00221511-0.00334593im
   0.00298565-1.97514e-5im   -0.00591451+7.63329e-5im
 -0.000645872+0.000375134im   0.00129193-0.000754078im
    0.0962443-0.165555im      -0.0962964+0.164296im
  -0.00078561+0.000383051im   0.00155457-0.000730383im
  0.000655863+0.000345015im  -0.00132876-0.00063683im
   -0.0356656+0.0176614im      0.0390892-0.0168105im
    -0.036109-0.0165017im      0.0399709+0.0144145im
   5.78959e-6-6.19387e-6im   -2.08244e-5+2.55114e-5im
    0.0326988+0.0206354im     -0.0332796-0.0226846im
 -0.000541353-0.000887014im   0.00113572+0.00173262im
             ⋮               
   2.94312e-5+5.2119e-5im    -0.00011559-0.000203064im
  0.000644112-0.000372913im  -0.00128395+0.000746449im
 -0.000642794+0.000373854im   0.00127962-0.000748977im
  0.000649258+0.000367152im   -0.0013039-0.00072299im
   -0.0305788+0.0168939im 

In [5]:
energy,V = eigen(Matrix(H))

Eigen{Float64, Float64, Matrix{Float64}, Vector{Float64}}
values:
336-element Vector{Float64}:
 -10.589851434307302
 -10.505395174664866
 -10.299101506582243
 -10.184275501256643
  -7.744153096456429
  -7.604027906884856
  -7.599011153956426
  -7.524959834272078
  -7.454862305297103
  -7.452121525270605
  -7.364715673058988
  -7.275770904159842
  -7.135477108421379
   ⋮
   7.806977595873471
   7.8093600343978595
   7.851170735708889
   7.9190877296102435
   7.95187556906168
   8.103391810943364
   8.14469098224083
   8.187385180858632
   8.209682556547396
   8.254046182852147
  10.752571281767525
  10.755895229236335
vectors:
336×336 Matrix{Float64}:
  0.575764     -5.34892e-15   0.808974     …  -2.25118e-19  -7.82308e-6
 -0.0101909     0.00739815    0.00559803      -1.69509e-6    8.82935e-6
 -0.0101909    -0.00739815    0.00559803       1.69509e-6    8.82935e-6
 -1.82257e-16   0.00105685   -2.4385e-17      -6.50723e-6    2.2714e-17
 -0.575764      0.404878      0.404487        -1.3454

In [35]:
dic = load_saved_dict(joinpath(folder_data, "meta_data_and_E.jld2"))

meta_data = dic["meta_data"]
sign_convention = :coordinate_first

U_values = meta_data["U_values"]
all_full_eig_vecs = dic["all_full_eig_vecs"]
all_E = dic["E"] # Needed for energy selection

indexer = dic["indexer"][2]

lattice = Square((3,2), Periodic())
subspace = HubbardSubspace(3, 2, lattice; k=indexer.k)
new_hopping, indexer = create_Hubbard(HubbardModel(1.0,0.0,0.0,false), subspace; get_indexer=true, momentum_basis=true, sign_convention=sign_convention)
new_interaction = create_Hubbard(HubbardModel(0.0,1.0,0.0,false), subspace; indexer=indexer, momentum_basis=true, sign_convention=sign_convention)
eigvals(Matrix(new_hopping +0.25 *new_interaction))[1]

In [ ]:
valid_files = [f for f in readdir(folder_new_sign) if occursin("HubbardED", f)]
file_path = joinpath(folder_new_sign, valid_files[1])
# println(file_path)
h5open(file_path, "r") do data
    N = (read(data, "metadata/nup"), read(data, "metadata/ndown"))
    spin_conserved = true
    use_symmetry = false

    Lvec = read(data, "metadata/Lvec")
    U_values = read(data, "data/uvec")
    kvecs = read(data, "metadata/kvecs")

    key_labels = [parse(Int, k) for k in keys(data["data/energies"])]
    all_E = [real.(read(data, "data/energies/$(k)"))[1, :] for k in key_labels] # Needed for energy selection
    k_min = find_best_energy_sector(all_E, U_values; labels=key_labels, data=data)
    println(kvecs)
    separate_spins = (read(data, "metadata/slater_labels/$k_min") isa Dict)
    if separate_spins
        sl_up = read(data, "metadata/slater_labels/$k_min/up")
        sl_dn = read(data, "metadata/slater_labels/$k_min/dn")
        println(sl_up)
        println(sl_dn)
        H_dim = size(sl_up, 2)
    else
        sl_all = read(data, "metadata/slater_labels/$k_min")
        H_dim = size(sl_all, 2)
    end

    global evecs_dataset = read(data, "data/evecs/$(k_min)")
end

H_hop, _, _ = Trotter.TamFermion.HubbardMomentumBasis(1.0, 0.0, [3,2], N_elec2; q_target=2)
H_int, _, _ = Trotter.TamFermion.HubbardMomentumBasis(0.0, 1.0, [3,2], N_elec2; q_target=2)

H = Matrix(H_hop + 0.25 * H_int)
state = evecs_dataset[:,2,2]
println(state'*H*state)
eigvals(H)[1]


In [ ]:
d = Trotter.fullSlaterMomBasis((3,2), 3, 2)
i = 100
println("up:$(d["qtot_up"][i]) down:$(d["qtot_dn"][i])")
string(d["ints"][100], base=2)


In [ ]:
abs.(vecs_new[1,:]'* vecs_new[2,:])

## 2. Compare Overlap & Energy Expectation vs U Spectrum

Examine how energy expectations of the Slater reference state and exact ground states change alongside ground state overlap as a function of interaction strength $U$.

In [ ]:
println("U (data)\tSlater E (data)\tGS E (data)\tOverlap (data)\t\tU (new)\t\tSlater E (new)\tGS E (new)\tOverlap (new)")
println("-"^115)
n_points = min(length(U1), length(U2))
for idx in 1:10:n_points
    u_val1 = U1[idx]
    u_val2 = U2[idx]
    
    gs1 = vecs_data[idx+1, :]
    gs2 = vecs_new[idx+1, :]
    
    e_slater1 = compute_energy_expectation(slater_data, u_val1, H_hop1, H_int1)
    e_gs1 = compute_energy_expectation(gs1, u_val1, H_hop1, H_int1)
    ov1 = abs(dot(slater_data, gs1))
    
    e_slater2 = compute_energy_expectation(slater_new, u_val2, H_hop2, H_int2)
    e_gs2 = compute_energy_expectation(gs2, u_val2, H_hop2, H_int2)
    ov2 = abs(dot(slater_new, gs2))
    
    println("$(round(u_val1, digits=3))\t\t$(round(e_slater1, digits=4))\t\t$(round(e_gs1, digits=4))\t\t$(round(ov1, digits=6))\t\t$(round(u_val2, digits=3))\t\t$(round(e_slater2, digits=4))\t\t$(round(e_gs2, digits=4))\t\t$(round(ov2, digits=6))")
end


## 3. Trotter Optimization Depth Scaling Experiment

Test Trotter optimization convergence for increasing depth (`num_exponentials` = 1, 2, 3, 4) on both datasets at a target $U$ index.

In [ ]:
function run_depth_scaling_comparison(folder::String; u_idx::Int=2, depths::Vector{Int}=[1, 2, 3], maxiters::Int=100, loss_type::Symbol=:overlap, antihermitian::Bool=false)
    U_vals, state_vecs, indexer, _, N_elec, _, _, _ = load_ED_data(
        folder; verbose=false, sign_convention=:spin_first, use_slater_reference="slater"
    )
    n_up, n_dn = N_elec
    Lvec = parse_lattice_dimension(folder)
    N_sites = prod(Lvec)
    
    basis_sector = Trotter.get_basis_sector(indexer, Lvec, N_sites)
    H_hop_sector, _, _ = Trotter.TamFermion.HubbardMomentumBasis(1.0, 0.0, Lvec, (n_up, n_dn); indexer=indexer)
    H_int_sector, _, _ = Trotter.TamFermion.HubbardMomentumBasis(0.0, 1.0, Lvec, (n_up, n_dn); indexer=indexer)
    
    gates = Trotter.enumerate_ferm_excitations(2, Lvec; conserve_mom=true, conserve_sz=true, include_diagonal=!antihermitian)
    tau_terms = Trotter.fgateToTauSector(gates, N_sites, basis_sector; antihermitian=antihermitian)
    
    target_state = state_vecs[u_idx, :]
    ref_state = state_vecs[1, :]
    u_val = U_vals[u_idx-1]
    
    target_energy = compute_energy_expectation(target_state, u_val, H_hop_sector, H_int_sector)
    ref_energy = compute_energy_expectation(ref_state, u_val, H_hop_sector, H_int_sector)
    
    println("Target U value: ", u_val, " (index ", u_idx-1, ")")
    println("Exact Ground State Energy: ", round(target_energy, digits=6))
    println("Reference State Energy:    ", round(ref_energy, digits=6))
    println("Initial Overlap Loss (un-optimized ref): ", round(1.0 - abs(dot(ref_state, target_state))^2, digits=6))
    
    results = Dict{Int, Float64}()
    for L in depths
        num_params = length(tau_terms) * L
        x0 = zeros(Float64, num_params)
        
        sol = optimize_unitary(
            x0, target_state, tau_terms, basis_sector, N_sites;
            maxiters=maxiters, loss_type=loss_type,
            H_hopping=H_hop_sector, H_interaction=H_int_sector,
            U=u_val, antihermitian=antihermitian
        )
        min_loss = sol.objective
        results[L] = min_loss
        
        opt_state = TrotterOptimization.apply_unitary(sol.u, gates, ref_state, basis_sector, N_sites, L; antihermitian=antihermitian)
        opt_energy = compute_energy_expectation(opt_state, u_val, H_hop_sector, H_int_sector)
        println("  Depth L=$L: final loss = ", round(min_loss, digits=8), " | Trotter State Energy = ", round(opt_energy, digits=6), " (dE = ", round(opt_energy - target_energy, digits=6), ")")
    end
    return results
end


### 3.1 Run Depth Scaling on `data` (JLD2)

In [ ]:
println("=== Depth Scaling Experiment: data (JLD2) ===")
res_data = run_depth_scaling_comparison(folder_data; u_idx=2, depths=[1, 2, 3], maxiters=100)

### 3.2 Run Depth Scaling on `data_new_sign` (HDF5)

In [ ]:
println("=== Depth Scaling Experiment: data_new_sign (HDF5) ===")
res_new = run_depth_scaling_comparison(folder_new_sign; u_idx=2, depths=[1, 2, 3], maxiters=100)

## 4. Custom Parameter Sandbox

Experiment with different parameters: change `folder`, `u_idx`, `loss_type` (`:overlap` or `:energy`), `antihermitian` (`true` or `false`), or `maxiters`.

In [ ]:
# Customize your run parameters here:
target_folder = folder_new_sign  # Change to folder_data or folder_new_sign
selected_u_idx = 2               # 2 corresponds to first U data point (after Slater ref)
num_exponentials = 2             # Number of Trotter layers
loss_function = :overlap         # Options: :overlap, :energy
use_antihermitian = false        # Options: true, false
max_iterations = 150             # Max iterations for L-BFGS

res_custom = run_depth_scaling_comparison(
    target_folder;
    u_idx=selected_u_idx,
    depths=[num_exponentials],
    maxiters=max_iterations,
    loss_type=loss_function,
    antihermitian=use_antihermitian
)